[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1aBIlvU4SwETuJqBPTmgW2jFp04DbQBXN)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

In [ ]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data/data.csv

In [ ]:
df = pd.read_csv("data.csv")

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df["level"])

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("xgb", XGBClassifier(eval_metric='logloss'))
])

param_dist = {
    'tfidf__min_df': randint(1, 5),
    'tfidf__max_df': uniform(0.7, 0.3),
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf__sublinear_tf': [True, False],

    'xgb__n_estimators': randint(50, 300),
    'xgb__max_depth': randint(3, 12),
    'xgb__learning_rate': uniform(0.01, 0.3),
    'xgb__subsample': uniform(0.6, 0.4),
    'xgb__colsample_bytree': uniform(0.5, 0.5),
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=30,
    scoring='accuracy',
    cv=3,
    verbose=4,
    n_jobs=-1,
    random_state=42
)

In [ ]:
random_search.fit(X_train_text, y_train)

print("Best params:", random_search.best_params_)
print("Best score:", random_search.best_score_)

In [ ]:
y_pred = random_search.best_estimator_.predict(X_test_text)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['viegls', 'vidējs', 'sarežģīts']))